# BioHub Cell Tracker — V0 submission

Baseline Kaggle complète : lecture Zarr, détection 3D, tracking, divisions prudentes, audit et `submission.csv`.

In [ ]:
from pathlib import Path
import os, sys, json, time

RUNTIME_CANDIDATES = [
    Path('/kaggle/input/biohub-cell-tracker-runtime'),
    Path('/kaggle/working/biohub-cell-tracker'),
    Path.cwd(),
    Path.cwd().parent,
]

def find_runtime_root():
    for candidate in RUNTIME_CANDIDATES:
        if (candidate / 'src/biohub_tracker').exists():
            return candidate
        if candidate.exists():
            for package_dir in candidate.rglob('src/biohub_tracker'):
                return package_dir.parent.parent
    return None

RUNTIME_ROOT = find_runtime_root()
if RUNTIME_ROOT is None:
    raise FileNotFoundError('Attach the biohub-cell-tracker runtime dataset to this notebook.')
sys.path.insert(0, str(RUNTIME_ROOT / 'src'))
print('Runtime:', RUNTIME_ROOT)

In [ ]:
from biohub_tracker.config import PipelineConfig
from biohub_tracker.pipeline import BaselinePipeline
from biohub_tracker.submission import audit_submission

COMPETITION = 'biohub-cell-tracking-during-development'
TEST_CANDIDATES = [
    Path(f'/kaggle/input/competitions/{COMPETITION}/test'),
    Path(f'/kaggle/input/{COMPETITION}/test'),
]
TEST_DIR = next((p for p in TEST_CANDIDATES if p.exists()), None)
if TEST_DIR is None:
    raise FileNotFoundError(f'Test directory not found. Checked: {TEST_CANDIDATES}')

config_path = RUNTIME_ROOT / 'configs/baseline.json'
config = PipelineConfig.from_json(config_path)
pipeline = BaselinePipeline(config)
print('Test:', TEST_DIR)
print('Config:', config)

In [ ]:
OUTPUT = Path('/kaggle/working/submission.csv')
STATS = Path('/kaggle/working/run_stats.csv')
started = time.perf_counter()
submission, stats = pipeline.run(TEST_DIR, OUTPUT)
stats.to_csv(STATS, index=False)
print(stats.to_string(index=False))
print(f'Elapsed: {(time.perf_counter() - started) / 60:.2f} min')

In [ ]:
report = audit_submission(submission)
print(json.dumps(report.to_dict(), indent=2))
if not report.valid:
    raise RuntimeError(report.errors)
if not OUTPUT.exists():
    raise RuntimeError('submission.csv was not written')
print(f'Ready: {OUTPUT} — {len(submission):,} rows')